In [ ]:
import pandas as pd
import re
import numpy as np
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm import tqdm

DetectorFactory.seed = 42

df = pd.read_csv('generic_dataset.csv')

print(f"Original shape: {df.shape}")
display(df.head(3))

Original shape: (1032225, 12)


,CommentID,VideoID,VideoTitle,AuthorName,AuthorChannelID,CommentText,Sentiment,Likes,Replies,PublishedAt,CountryCode,CategoryID
0,UgyRjrEdJIPrf68uND14AaABAg,mcY4M9gjtsI,They killed my friend.#tales #movie #shorts,@OneWhoWandered,UC_-UEXaBL1dqqUPGkDll49A,Anyone know what movie this is?,Neutral,0,2,2025-01-15 00:54:55,NZ,1
1,UgxXxEIySAwnMNw8D7N4AaABAg,2vuXcw9SZbA,Man Utd conceding first penalty at home in yea...,@chiefvon3068,UCZ1LcZESjYqzaQRhjdZJFwg,The fact they're holding each other back while...,Positive,0,0,2025-01-13 23:51:46,AU,17
2,UgxB0jh2Ur41mcXr5IB4AaABAg,papg2tsoFzg,Welcome to Javascript Course,@Abdulla-ip8qr,UCWBK35w5Swy1iF5xIbEyw3A,waiting next video will be?,Neutral,1,0,2020-07-06 13:18:16,IN,27


In [ ]:
# Drop any completely empty rows immediately
df = df.dropna(subset=['CommentText', 'Sentiment']).copy()

# Ensure the comments are actually strings
df['CommentText'] = df['CommentText'].astype(str)

# Standardize the sentiment labels to 0, 1, 2 if they aren't already
sentiment_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
if df['Sentiment'].dtype == 'object':
    df['Sentiment'] = df['Sentiment'].map(sentiment_map)

In [ ]:
# 1. Word Count Filter (5–50 words)
df['word_count'] = df['CommentText'].str.split().str.len()
mask_length = (df['word_count'] >= 5) & (df['word_count'] <= 50)

# 2. Spam Filter
spam_pattern = r'(http|www\.|\.com|\.org|\.net)'
mask_no_spam = ~df['CommentText'].str.contains(spam_pattern, case=False, na=False, regex=True)

# 3. Timestamp Filter (NEW)
timestamp_pattern = r'\b\d{1,2}:\d{2}(:\d{2})?\b'
mask_no_timestamp = ~df['CommentText'].str.contains(timestamp_pattern, regex=True, na=False)

# 4. Pure Timestamp Line Removal (optional but strong)
mask_not_pure_timestamp = ~df['CommentText'].str.match(r'^\s*\d{1,2}:\d{2}(:\d{2})?\s*$', na=False)

print("Surface filters prepared (emoji-safe).")

C:\Users\addys\AppData\Local\Temp\ipykernel_40756\732892159.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_no_spam = ~df['CommentText'].str.contains(spam_pattern, case=False, na=False, regex=True)
C:\Users\addys\AppData\Local\Temp\ipykernel_40756\732892159.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_no_timestamp = ~df['CommentText'].str.contains(timestamp_pattern, regex=True, na=False)


Surface filters prepared (emoji-safe).


In [17]:
# %%
clean_df = df[
    mask_length &
    mask_no_spam &
    mask_no_timestamp &
    mask_not_pure_timestamp
].copy()

clean_df = clean_df.drop(columns=['word_count'])

print(f"Shape after removing surface noise: {clean_df.shape}")
print("Remaining sentiment distribution:")
print(clean_df['Sentiment'].value_counts())

Shape after removing surface noise: (782242, 12)
Remaining sentiment distribution:
Sentiment
0    279453
2    255331
1    247458
Name: count, dtype: int64


In [18]:
# %%
def is_english(text):
    try:
        return detect(text) == 'en'
    except LangDetectException:
        # ⚠️ CHANGE: Keep emoji-heavy comments instead of dropping
        return True

print("Running deep language detection...")

tqdm.pandas(desc="Detecting English")
mask_true_english = clean_df['CommentText'].progress_apply(is_english)

clean_df = clean_df[mask_true_english].copy()

print(f"\nShape after strict English filtering: {clean_df.shape}")
print("Remaining sentiment distribution:")
print(clean_df['Sentiment'].value_counts())

Running deep language detection...


Detecting English:   0%|          | 0/782242 [00:00<?, ?it/s]

Detecting English: 100%|██████████| 782242/782242 [41:46<00:00, 312.06it/s]   



Shape after strict English filtering: (716895, 12)
Remaining sentiment distribution:
Sentiment
0    258559
2    235701
1    222635
Name: count, dtype: int64


In [19]:
# %%
sampled_df = clean_df.groupby('Sentiment').sample(n=16000, random_state=42)
 
final_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final Downsampled Shape: {final_df.shape}")
print("Final Sentiment Distribution:")
print(final_df['Sentiment'].value_counts())

Final Downsampled Shape: (48000, 12)
Final Sentiment Distribution:
Sentiment
2    16000
0    16000
1    16000
Name: count, dtype: int64


In [20]:
# %%
final_df.to_csv('youtube_comments_48k_clean.csv', index=False)
print("Saved to 'youtube_comments_48k_clean.csv'. Data cleaning complete!")

Saved to 'youtube_comments_48k_clean.csv'. Data cleaning complete!


In [1]:
import pandas as pd
df = pd.read_csv("youtube_comments_48k_clean.csv")

In [2]:
# Count comments containing newline or carriage return
mask_newlines = df['CommentText'].str.contains(r'[\n\r]', regex=True, na=False)
num_affected = mask_newlines.sum()

print(f"Number of comments affected by newline/carriage return cleaning: {num_affected}")

Number of comments affected by newline/carriage return cleaning: 0


In [4]:
df['CommentText'] = df['CommentText'].str.replace(r'[\n\r]+', ' ', regex=True)

In [ ]:
# %%
df.to_csv('youtube_comments_48k_removed_\n.csv', index=False)
print("Saved to 'youtube_comments_48k_clean.csv'. Data cleaning complete!")

Saved to 'youtube_comments_48k_clean.csv'. Data cleaning complete!
